In [1]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import re
import json
import random
import importlib.util
import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
TRAIN_PATH = "train_data/subtask 1/train_data.json"
TEST_PATH = "test_data/subtask 1/test_data_subtask_1.json"
OUTPUT_PATH = "kcast_qwen25_7b_best_predictions.json"
TMP_PRED_PATH = "kcast_tmp_predictions.json"
TMP_SCORE_PATH = "kcast_tmp_score.json"
EVAL_SCRIPT_PATH = "evaluation_kit/task 1 & 3/evaluation_script.py"

ALPHA_CANDIDATES  = [0,  -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]
K = 15
LAYER_START_FRAC = 0.75
MAX_LEN = 1024
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
SEED = 42
PRINT_EVERY_TRAIN = 100
PRINT_EVERY_TEST = 50

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

def load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(x, p):
    with open(p, "w", encoding="utf-8") as f:
        json.dump(x, f, indent=2, ensure_ascii=False)

def norm(x):
    return re.sub(r"\s+", " ", x).strip()

def build_prompt(tok, s):
    msg = [{
        "role": "user",
        "content": "Determine whether the following syllogism is formally valid. Ignore real-world knowledge and plausibility. Judge only by logical structure. Answer with exactly one word: VALID or INVALID.\n\nSyllogism: " + norm(s)
    }]
    return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

def get_layers(m):
    if hasattr(m, "model") and hasattr(m.model, "layers"):
        return m.model.layers
    if hasattr(m, "transformer") and hasattr(m.transformer, "h"):
        return m.transformer.h
    raise ValueError("Unsupported model architecture")

def tok_ids(tok):
    v = tok.encode(" VALID", add_special_tokens=False)
    i = tok.encode(" INVALID", add_special_tokens=False)
    if len(v) == 0:
        v = tok.encode("VALID", add_special_tokens=False)
    if len(i) == 0:
        i = tok.encode("INVALID", add_special_tokens=False)
    return v[0], i[0]

def pos(mask):
    return int(mask.sum(dim=1)[0].item()) - 1

def forward(model, tok, prompt):
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    p = pos(enc["attention_mask"])
    hs = [out.hidden_states[i][0, p, :].detach().float().cpu() for i in range(1, len(out.hidden_states))]
    logits = out.logits[0, p, :].detach().float().cpu()
    return hs, logits

def predict(model, tok, prompt, v_id, i_id):
    hs, logits = forward(model, tok, prompt)
    return logits[v_id].item() > logits[i_id].item(), hs

def mean_layerwise(v):
    return [torch.stack([x[i] for x in v], dim=0).mean(0) for i in range(len(v[0]))]

def unit(x):
    return x / (x.norm() + 1e-8)

def load_eval_runner(path):
    spec = importlib.util.spec_from_file_location("semeval_eval_module", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.run_full_scoring

def hook_fn(delta_layer, alpha, p):
    def h(module, inputs):
        x = inputs[0]
        idx = x.shape[1] - 1 if p < 0 else p
        x = x.clone()
        x[:, idx, :] = x[:, idx, :] + alpha * delta_layer.to(x.device, dtype=x.dtype)
        return (x,)
    return h

def steered_predict(model, tok, prompt, v_id, i_id, layers, sel, delta, knn, y, alpha):
    hs0, _ = forward(model, tok, prompt)
    q = torch.cat([hs0[l] for l in sel], dim=0).numpy()[None, :]
    idx = knn.kneighbors(q, return_distance=False)[0]
    yhat = 1 if y[idx].sum() >= 0 else -1
    signed_alpha = -yhat * alpha

    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    p = pos(enc["attention_mask"])

    handles = []
    for i, l in enumerate(sel):
        handles.append(layers[l].register_forward_pre_hook(hook_fn(delta[i], signed_alpha, p)))

    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)

    for h in handles:
        h.remove()

    logits = out.logits[0, p, :].detach().float().cpu()
    pred = logits[v_id].item() > logits[i_id].item()
    return pred

print("Loading tokenizer and model...")
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    output_hidden_states=True
).to(DEVICE)
model.eval()

layers = get_layers(model)
start = int(len(layers) * LAYER_START_FRAC)
sel = list(range(start, len(layers)))
v_id, i_id = tok_ids(tok)

print("Loading data...")
train = load_json(TRAIN_PATH)
test = load_json(TEST_PATH)

print("Collecting training activations...")
correct = []
wrong = []
records = []

for idx, ex in enumerate(train):
    if idx % PRINT_EVERY_TRAIN == 0:
        print(f"train progress {idx}/{len(train)}")
    pr = build_prompt(tok, ex["syllogism"])
    pred, hs = predict(model, tok, pr, v_id, i_id)
    gold = bool(ex["validity"])
    h = [hs[l] for l in sel]
    if pred == gold:
        correct.append(h)
    else:
        wrong.append(h)
    records.append({
        "hidden": h,
        "label": 1 if gold else -1
    })

if len(correct) == 0 or len(wrong) == 0:
    raise ValueError("Need both correct and wrong training examples to compute delta.")

print("Computing steering vector...")
mu_p = mean_layerwise(correct)
mu_n = mean_layerwise(wrong)
delta = [unit(mu_p[i] - mu_n[i]) for i in range(len(mu_p))]

print("Building kNN index...")
X = []
y = []
for r in records:
    X.append(torch.cat(r["hidden"], dim=0).numpy())
    y.append(r["label"])
X = np.stack(X, axis=0)
y = np.array(y)

knn = NearestNeighbors(n_neighbors=min(K, len(X)), metric="cosine")
knn.fit(X)

print("Loading evaluation runner...")
run_full_scoring = load_eval_runner(EVAL_SCRIPT_PATH)

best_alpha = None
best_score = None
best_preds = None

for alpha in ALPHA_CANDIDATES:
    print(f"\nRunning alpha {alpha}")
    preds = []
    for idx, ex in enumerate(test):
        if idx % PRINT_EVERY_TEST == 0:
            print(f"alpha {alpha} test progress {idx}/{len(test)}")
        pr = build_prompt(tok, ex["syllogism"])
        pred = steered_predict(model, tok, pr, v_id, i_id, layers, sel, delta, knn, y, alpha)
        preds.append({"id": ex["id"], "validity": bool(pred)})

    save_json(preds, TMP_PRED_PATH)
    run_full_scoring(TEST_PATH, TMP_PRED_PATH, TMP_SCORE_PATH)
    score = load_json(TMP_SCORE_PATH)

    acc = score.get("accuracy")
    ce = score.get("content_effect")
    comb = score.get("combined_score")

    print(f"alpha={alpha} accuracy={acc} content_effect={ce} combined_score={comb}")

    if best_score is None or comb > best_score:
        best_score = comb
        best_alpha = alpha
        best_preds = preds

save_json(best_preds, OUTPUT_PATH)

print("\nFinished.")
print(f"best_alpha={best_alpha}")
print(f"best_combined_score={best_score}")
print(f"saved_best_predictions={OUTPUT_PATH}")

Loading tokenizer and model...


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading data...
train progress 0/960
train progress 100/960
train progress 200/960
train progress 300/960
train progress 400/960
train progress 500/960
train progress 600/960
train progress 700/960
train progress 800/960
train progress 900/960
Computing steering vector...
Building kNN index...
Loading evaluation runner...

Running alpha 0
alpha 0 test progress 0/191
alpha 0 test progress 50/191
alpha 0 test progress 100/191
alpha 0 test progress 150/191
Scoring complete. Results written to kcast_tmp_score.json
alpha=0 accuracy=68.0628 content_effect=39.5833 combined_score=14.4711

Running alpha -3
alpha -3 test progress 0/191
alpha -3 test progress 50/191
alpha -3 test progress 100/191
alpha -3 test progress 150/191
Scoring complete. Results written to kcast_tmp_score.json
alpha=-3 accuracy=69.6335 content_effect=39.805 combined_score=14.7879

Running alpha -2
alpha -2 test progress 0/191
alpha -2 test progress 50/191
alpha -2 test progress 100/191
alpha -2 test progress 150/191
Scorin

In [1]:
import os
import re
import json
import random
import importlib.util
import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
TRAIN_PATH = "train_data/subtask 1/train_data.json"
TEST_PATH = "test_data/subtask 1/test_data_subtask_1.json"
OUTPUT_PATH = "kcast_qwen25_7b_best_predictions.json"
TMP_PRED_PATH = "kcast_tmp_predictions.json"
TMP_SCORE_PATH = "kcast_tmp_score.json"
EVAL_SCRIPT_PATH = "evaluation_kit/task 1 & 3/evaluation_script.py"

ALPHA_CANDIDATES = [0, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 7, 10]
K = 15
LAYER_START_FRAC = 0.66
KNN_MODE = "per_layer"
MAX_LEN = 1024
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
SEED = 42
PRINT_EVERY_TRAIN = 100
PRINT_EVERY_TEST = 50

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

def load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(x, p):
    with open(p, "w", encoding="utf-8") as f:
        json.dump(x, f, indent=2, ensure_ascii=False)

def norm(x):
    return re.sub(r"\s+", " ", x).strip()

def build_prompt(tok, s):
    msg = [{
        "role": "user",
        "content": "Determine whether the following syllogism is formally valid. Ignore real-world knowledge and plausibility. Judge only by logical structure. Answer with exactly one word: VALID or INVALID.\n\nSyllogism: " + norm(s)
    }]
    return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

def get_layers(m):
    if hasattr(m, "model") and hasattr(m.model, "layers"):
        return m.model.layers
    if hasattr(m, "transformer") and hasattr(m.transformer, "h"):
        return m.transformer.h
    raise ValueError("Unsupported model architecture")

def tok_ids(tok):
    v = tok.encode(" VALID", add_special_tokens=False)
    i = tok.encode(" INVALID", add_special_tokens=False)
    if len(v) == 0:
        v = tok.encode("VALID", add_special_tokens=False)
    if len(i) == 0:
        i = tok.encode("INVALID", add_special_tokens=False)
    return v[0], i[0]

def pos(mask):
    return int(mask.sum(dim=1)[0].item()) - 1

def forward(model, tok, prompt):
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)
    p = pos(enc["attention_mask"])
    hs = [out.hidden_states[i][0, p, :].detach().float().cpu() for i in range(1, len(out.hidden_states))]
    logits = out.logits[0, p, :].detach().float().cpu()
    return hs, logits

def predict(model, tok, prompt, v_id, i_id):
    hs, logits = forward(model, tok, prompt)
    return logits[v_id].item() > logits[i_id].item(), hs

def mean_layerwise(v):
    return [torch.stack([x[i] for x in v], dim=0).mean(0) for i in range(len(v[0]))]

def unit(x):
    return x / (x.norm() + 1e-8)

def load_eval_runner(path):
    spec = importlib.util.spec_from_file_location("semeval_eval_module", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.run_full_scoring

def output_hook_fn(delta_layer, alpha, p):
    def h(module, inputs, output):
        if isinstance(output, tuple):
            x = output[0].clone()
            d = delta_layer.to(x.device, dtype=x.dtype)
            x[:, p, :] = x[:, p, :] + alpha * d
            return (x,) + output[1:]
        x = output.clone()
        d = delta_layer.to(x.device, dtype=x.dtype)
        x[:, p, :] = x[:, p, :] + alpha * d
        return x
    return h

def build_knn_concat(records):
    X = []
    y = []
    for r in records:
        X.append(torch.cat(r["hidden"], dim=0).numpy())
        y.append(r["label"])
    X = np.stack(X, axis=0)
    y = np.array(y)
    knn = NearestNeighbors(n_neighbors=min(K, len(X)), metric="cosine")
    knn.fit(X)
    return knn, y

def build_knn_per_layer(records, n_layers):
    knn_list = []
    y_list = []
    for li in range(n_layers):
        X = []
        y = []
        for r in records:
            X.append(r["hidden"][li].numpy())
            y.append(r["label"])
        X = np.stack(X, axis=0)
        y = np.array(y)
        knn = NearestNeighbors(n_neighbors=min(K, len(X)), metric="cosine")
        knn.fit(X)
        knn_list.append(knn)
        y_list.append(y)
    return knn_list, y_list

def predict_condition_concat(hs0, sel, knn, y):
    q = torch.cat([hs0[l] for l in sel], dim=0).numpy()[None, :]
    idx = knn.kneighbors(q, return_distance=False)[0]
    return 1 if y[idx].sum() >= 0 else -1

def predict_condition_per_layer(hs0, sel, knn_list, y_list):
    votes = []
    for i, l in enumerate(sel):
        q = hs0[l].numpy()[None, :]
        idx = knn_list[i].kneighbors(q, return_distance=False)[0]
        yhat = 1 if y_list[i][idx].sum() >= 0 else -1
        votes.append(yhat)
    s = sum(votes)
    return 1 if s >= 0 else -1

def steered_predict(model, tok, prompt, v_id, i_id, layers, sel, delta, alpha, knn=None, y=None, knn_list=None, y_list=None):
    hs0, _ = forward(model, tok, prompt)

    if KNN_MODE == "concat":
        yhat = predict_condition_concat(hs0, sel, knn, y)
    else:
        yhat = predict_condition_per_layer(hs0, sel, knn_list, y_list)

    signed_alpha = -yhat * alpha

    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    p = pos(enc["attention_mask"])

    handles = []
    for i, l in enumerate(sel):
        handles.append(layers[l].register_forward_hook(output_hook_fn(delta[i], signed_alpha, p)))

    with torch.no_grad():
        out = model(**enc, output_hidden_states=True, use_cache=False)

    for h in handles:
        h.remove()

    logits = out.logits[0, p, :].detach().float().cpu()
    pred = logits[v_id].item() > logits[i_id].item()
    return pred

print("Loading tokenizer and model...")
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE
).to(DEVICE)
model.eval()

layers = get_layers(model)
start = int(len(layers) * LAYER_START_FRAC)
sel = list(range(start, len(layers)))
v_id, i_id = tok_ids(tok)

print("Selected layers:", sel)

print("Loading data...")
train = load_json(TRAIN_PATH)
test = load_json(TEST_PATH)

print("Collecting training activations...")
correct = []
wrong = []
records = []

for idx, ex in enumerate(train):
    if idx % PRINT_EVERY_TRAIN == 0:
        print(f"train progress {idx}/{len(train)}")
    pr = build_prompt(tok, ex["syllogism"])
    pred, hs = predict(model, tok, pr, v_id, i_id)
    gold = bool(ex["validity"])
    h = [hs[l] for l in sel]
    if pred == gold:
        correct.append(h)
    else:
        wrong.append(h)
    records.append({
        "hidden": h,
        "label": 1 if gold else -1
    })

if len(correct) == 0 or len(wrong) == 0:
    raise ValueError("Need both correct and wrong training examples to compute delta.")

print("Computing steering vector...")
mu_p = mean_layerwise(correct)
mu_n = mean_layerwise(wrong)
delta = [unit(mu_p[i] - mu_n[i]) for i in range(len(mu_p))]

print("Building kNN index...")
if KNN_MODE == "concat":
    knn, y = build_knn_concat(records)
    knn_list, y_list = None, None
else:
    knn_list, y_list = build_knn_per_layer(records, len(sel))
    knn, y = None, None

print("Loading evaluation runner...")
run_full_scoring = load_eval_runner(EVAL_SCRIPT_PATH)

best_alpha = None
best_score = None
best_preds = None

for alpha in ALPHA_CANDIDATES:
    print(f"\nRunning alpha {alpha}")
    preds = []
    for idx, ex in enumerate(test):
        if idx % PRINT_EVERY_TEST == 0:
            print(f"alpha {alpha} test progress {idx}/{len(test)}")
        pr = build_prompt(tok, ex["syllogism"])
        pred = steered_predict(
            model,
            tok,
            pr,
            v_id,
            i_id,
            layers,
            sel,
            delta,
            alpha,
            knn=knn,
            y=y,
            knn_list=knn_list,
            y_list=y_list
        )
        preds.append({"id": ex["id"], "validity": bool(pred)})

    save_json(preds, TMP_PRED_PATH)
    run_full_scoring(TEST_PATH, TMP_PRED_PATH, TMP_SCORE_PATH)
    score = load_json(TMP_SCORE_PATH)

    acc = score.get("accuracy")
    ce = score.get("content_effect")
    comb = score.get("combined_score")

    print(f"alpha={alpha} accuracy={acc} content_effect={ce} combined_score={comb}")

    if best_score is None or comb > best_score:
        best_score = comb
        best_alpha = alpha
        best_preds = preds

save_json(best_preds, OUTPUT_PATH)

print("\nFinished.")
print(f"best_alpha={best_alpha}")
print(f"best_combined_score={best_score}")
print(f"saved_best_predictions={OUTPUT_PATH}")

Loading tokenizer and model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Selected layers: [18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Loading data...
train progress 0/960
train progress 100/960
train progress 200/960
train progress 300/960
train progress 400/960
train progress 500/960
train progress 600/960
train progress 700/960
train progress 800/960
train progress 900/960
Computing steering vector...
Building kNN index...
Loading evaluation runner...

Running alpha 0
alpha 0 test progress 0/191
alpha 0 test progress 50/191
alpha 0 test progress 100/191
alpha 0 test progress 150/191
Scoring complete. Results written to kcast_tmp_score.json
alpha=0 accuracy=68.5864 content_effect=38.5417 combined_score=14.6635

Running alpha -3
alpha -3 test progress 0/191
alpha -3 test progress 50/191
alpha -3 test progress 100/191
alpha -3 test progress 150/191
Scoring complete. Results written to kcast_tmp_score.json
alpha=-3 accuracy=69.6335 content_effect=43.9938 combined_score=14.4873

Running alpha -2
alpha -2 test progress 0/191
alpha -2 test progress 50/191
alpha -2 

In [4]:
pip install -U typing_extensions

Note: you may need to restart the kernel to use updated packages.
